## Imports

In [137]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd
import numpy as np

## Loading and Viewing the Datasets

In [138]:
# creating a variable to store the absolute .csv file path for the full Deep4Chem dataset
d4c_path = r'C:/Users/Amy/Desktop/Masters_Project/paper_methods/Deep4Chem_chemprop/Deep4Chem_chemprop/d4c_ext_coef_all_data.csv'

# reading in the dataset as stored in the path variable
d4c = pd.read_csv(d4c_path)

# printing the index and heading of each column in the dataset
for i, col in enumerate(d4c.columns):
    print(i, repr(col))

# viewing the shape (no. entries) and title of columns contained by the data
print('Shape:', d4c.shape)
print('\nColumns:')
for col in d4c.columns:
    print(repr(col))

# checking for missing values within the dataset
print('\nMissing values:')
print(d4c.isnull().sum())

0 'Chromophore'
1 'Solvent'
2 'Absorption max (nm)'
3 'log(e/mol-1 dm3 cm-1)'
Shape: (8032, 4)

Columns:
'Chromophore'
'Solvent'
'Absorption max (nm)'
'log(e/mol-1 dm3 cm-1)'

Missing values:
Chromophore              0
Solvent                  0
Absorption max (nm)      0
log(e/mol-1 dm3 cm-1)    0
dtype: int64


In [139]:
# dropping invalid solvent SMILES strings from the full Deep4Chem dataset - the only obviously invalid entries are those named 'gas', so they are
# explicitly dropped here
d4c = d4c[d4c['Solvent'].str.lower() != 'gas'].reset_index(drop=True)

In [140]:
# viewing the shape and head of the dataset following the removal of the invalid solvent entries ('gas')
print(d4c.shape)
d4c.head()

(8018, 4)


,Chromophore,Solvent,Absorption max (nm),log(e/mol-1 dm3 cm-1)
0,c1ccc2ccccc2c1,C1CCCCC1,286.0,3.58
1,C[Si](C)(C)c1cccc2ccccc12,C1CCCCC1,294.0,3.73
2,C[SiH](C)c1cccc2ccccc12,C1CCCCC1,294.0,3.75
3,CCCC[Si](C)(C)c1cccc2ccccc12,C1CCCCC1,294.0,3.74
4,CC(C)(C)[Si](C)(C)c1cccc2ccccc12,C1CCCCC1,295.0,3.78


In [141]:
# creating a variable to store the absolute .csv file path for the Deep4Chem training subset
d4c_train_path = r'C:/Users/Amy/Desktop/Masters_Project/paper_methods/d4c_train.csv'

# reading in the training subset as stored in the path variable
d4c_train = pd.read_csv(d4c_train_path)

In [142]:
# creating a variable to store the absolute .csv file path for the Deep4Chem test subset
d4c_test_path = r'C:/Users/Amy/Desktop/Masters_Project/paper_methods/d4c_test.csv'

# reading in the test subset as stored in the path variable
d4c_test = pd.read_csv(d4c_test_path)

## Descriptor Generation for Full Deep4Chem Dataset

In [143]:
# retrieving the full list of available RDKit molecular descriptor functions
descriptor_functions = Descriptors.descList

# extracting the name of all descriptors into a list, and printing the number of descriptors contained within the list
descriptor_names = [name for name, func in descriptor_functions]
print(len(descriptor_names))

208


In [144]:
# defining a function to calculate RDKit molecular descriptors from SMILES input strings
def calculate_descriptors(smiles):
    # converting SMILES string into a RDKitn molecule object
    mol = Chem.MolFromSmiles(smiles)
    # returning a list of missing values if any SMILES strings are invalid
    if mol is None:
        return [np.nan] * len(descriptor_functions)

    # initialising a list to store descriptor values
    descriptor_values = []

    # iterating through each RDKit descriptor
    for name, func in descriptor_functions:
        # calculating and storing the descriptor values in the previously initialised list - storing missing values if descriptors cannot be calculated
        try:
            descriptor_values.append(func(mol))
        except Exception:
            descriptor_values.append(np.nan)

    # returning the complete list of descriptor values for the molecule corresponding to the input SMILES string
    return descriptor_values

In [145]:
# initialising a list to store descriptor vectors for all chromophores
chromophore_desc = []

# iterating through every chromophore SMILES in the dataset, and calculating and storing all corresponding descriptors for each molecule
for smiles in d4c['Chromophore']:
    chromophore_desc.append(calculate_descriptors(smiles))

# converting the chromophore descriptor lists into a DataFrame - prefixing each descriptor name with 'Chrom_' to distinguish chromophore descriptors 
# from solvent descriptors (calculated in a following cell)
chromophore_df = pd.DataFrame(chromophore_desc, columns=[f'Chrom_{d}' for d in descriptor_names])

In [146]:
# viewing the length of the full Deep4Chem database following conversion of the chromophore input SMILES strings into RDKit molecular descriptors to
# ensure the process was successful on every valid row - checking the data type of the first (chromophore SMILES) entry of the descriptor vector, and
# all 208 of its values
print(len(chromophore_desc))
print(type(chromophore_desc[0]))
print(chromophore_desc[0])

8018
<class 'list'>
[2.1203703703703702, 1.3101851851851853, 2.1203703703703702, 1.3101851851851853, 0.5114311994891171, 128.17399999999995, 120.10999999999997, 128.062600256, 48, 0, -0.018404250672442266, -0.06162971144687501, 0.06162971144687501, 0.018404250672442266, 0.5, 0.8, 1.1, 13.897200748817012, 10.132094340785901, 1.8457978667862491, -1.919798598723747, 2.1057162842241914, -1.6654553018517613, 5.8148423611800535, 1.7511038765669886, 2.8880516016579914, 271.01518240905483, 6.811554787871634, 5.618802153517007, 5.618802153517007, 4.9663264951887856, 3.404700538379253, 3.404700538379253, 2.3471506281091274, 2.3471506281091274, 1.6586892478083903, 1.6586892478083903, 1.132905824745182, 1.132905824745182, -1.2999999999999998, 348.57046183703704, 5.482229779997874, 2.143052886105772, 0.7805335436910606, 60.11306765123217, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 48.53093654769288, 10.772448428929591, 0.0, 0.0, 0.0, 10.772448428929591, 0.0, 0.0, 0.0, 0.0, 0.0, 48.5309365476

In [147]:
# initialising a list to store descriptor vectors for all solvents
solvent_desc = []

# iterating through every solvent SMILES in the dataset, and calculating and storing all corresponding descriptors for each molecule
for smiles in d4c['Solvent']:
    solvent_desc.append(calculate_descriptors(smiles))

# converting the solvent descriptor lists into a DataFrame - prefixing each descriptor name with 'Solv_' to distinguish solvent descriptors from 
# chromophore descriptors (calculated in a previous cell)
solvent_df = pd.DataFrame(solvent_desc, columns=[f'Solv_{d}' for d in descriptor_names])

In [148]:
# viewing the length of the full Deep4Chem database following conversion of the solvent input SMILES strings into RDKit molecular descriptors to ensure
# the process was successful on every valid row - checking the data type of the first (solvent SMILES) entry of the descriptor vector, and all 208 of 
# its values
print(len(solvent_desc))
print(type(solvent_desc[0]))
print(solvent_desc[0])

8018
<class 'list'>
[1.5, 1.5, 1.5, 1.5, 0.42231618686094674, 84.162, 72.06599999999999, 84.093900384, 36, 0, -0.0533059727891157, -0.0533059727891157, 0.0533059727891157, 0.0533059727891157, 0.3333333333333333, 0.5, 0.6666666666666666, 14.014, 10.012, 1.9496940272108838, -2.052305972789116, 2.1470999999999996, -1.8548999999999998, 4.506, 0.5039999999999998, 2.0, 15.509775004326936, 4.242640687119286, 4.242640687119285, 4.242640687119285, 3.0, 2.9999999999999996, 2.9999999999999996, 2.121320343559642, 2.121320343559642, 1.4999999999999996, 1.4999999999999996, 1.0606601717798207, 1.0606601717798207, 0.0, 34.3994618804395, 4.166666666666667, 2.2222222222222223, 1.0, 39.55813044628337, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 38.524929737556064, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 38.524929737556064, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 38.524929737556064, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 38.524929737556064, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

In [149]:
# concatenating both chromophore and solvent descriptor DataFrames as a matrix - each column are the descriptor names for both chromophore and solvent
# inputs, and each row is the corresponding values calculated for each and every (chromophore and solvent) molecular descriptor
X = pd.concat([chromophore_df, solvent_df], axis=1)

In [150]:
# viewing the shape and first few values of this descriptor matrix to ensure all valid entries in the original full Deep4Chem dataset have been 
# retained, and that the descriptors concatenated correctly
print(X.shape)
print(X.head())

(8018, 416)
   Chrom_MaxEStateIndex  Chrom_MinEStateIndex  Chrom_MaxAbsEStateIndex  \
0              2.120370              1.310185                 2.120370   
1              2.397593             -1.197755                 2.397593   
2              2.376759             -0.679074                 2.376759   
3              2.504398             -1.264830                 2.504398   
4              2.480926             -1.445440                 2.480926   

   Chrom_MinAbsEStateIndex  Chrom_qed  Chrom_MolWt  Chrom_HeavyAtomMolWt  \
0                 1.310185   0.511431      128.174               120.110   
1                 1.197755   0.619051      200.357               184.229   
2                 0.679074   0.600400      186.330               172.218   
3                 1.264830   0.689738      242.438               220.262   
4                 0.386036   0.643330      242.438               220.262   

   Chrom_ExactMolWt  Chrom_NumValenceElectrons  Chrom_NumRadicalElectrons  \
0        

In [151]:
# creating a list containing only the target property columns from the full Deep4Chem dataset
y = d4c[['Absorption max (nm)', 'log(e/mol-1 dm3 cm-1)']]

In [152]:
# viewing the shape and first few values of this list to ensure all valid entries in the original full Deep4Chem dataset have been retained, and that 
# the dataset was reduced correctly
print(y.shape)
print(y.head())

(8018, 2)
   Absorption max (nm)  log(e/mol-1 dm3 cm-1)
0                286.0                   3.58
1                294.0                   3.73
2                294.0                   3.75
3                294.0                   3.74
4                295.0                   3.78


In [153]:
# creating a Boolean mask that is 'False' for rows containing any missing values, and 'True' for rows with no missing values within the descriptor
# matrix
mask = ~X.isna().any(axis=1)

# keeping only the rows of the descriptor matrix and target properties with no missing descriptor values
X = X.loc[mask]
y = y.loc[mask]

# identically filtering the full Deep4Chem dataset, ensuring it contains only molecules with valid descriptors and corresponding target values
d4c = d4c.loc[mask]

In [154]:
# viewing the lengths of the descriptor matrix and target properties list to ensure they are the same length following data cleaning
print(len(X))
print(len(y))

7996
7996


In [155]:
# constructing the fully-featurised Deep4Chem dataset by concatenating the descriptor matrix and the measured target property values list - the first
# 416 columns correspond to the former's features, and the final 2 columns correspond to the latter's
descriptor_dataset = pd.concat([X, y], axis=1)

# saving the featurised Deep4Chem dataset as a CSV file
descriptor_dataset.to_csv('d4c_rdkit_descriptors.csv', index=False)

In [156]:
# viewing the shape of the fully-featurised Deep4Chem dataset
print('Shape:', descriptor_dataset.shape)

# iterating through and printing the title of every column within the descriptor dataset
print('\nColumns:')
for col in descriptor_dataset.columns:
    print(repr(col))

Shape: (7996, 418)

Columns:
'Chrom_MaxEStateIndex'
'Chrom_MinEStateIndex'
'Chrom_MaxAbsEStateIndex'
'Chrom_MinAbsEStateIndex'
'Chrom_qed'
'Chrom_MolWt'
'Chrom_HeavyAtomMolWt'
'Chrom_ExactMolWt'
'Chrom_NumValenceElectrons'
'Chrom_NumRadicalElectrons'
'Chrom_MaxPartialCharge'
'Chrom_MinPartialCharge'
'Chrom_MaxAbsPartialCharge'
'Chrom_MinAbsPartialCharge'
'Chrom_FpDensityMorgan1'
'Chrom_FpDensityMorgan2'
'Chrom_FpDensityMorgan3'
'Chrom_BCUT2D_MWHI'
'Chrom_BCUT2D_MWLOW'
'Chrom_BCUT2D_CHGHI'
'Chrom_BCUT2D_CHGLO'
'Chrom_BCUT2D_LOGPHI'
'Chrom_BCUT2D_LOGPLOW'
'Chrom_BCUT2D_MRHI'
'Chrom_BCUT2D_MRLOW'
'Chrom_BalabanJ'
'Chrom_BertzCT'
'Chrom_Chi0'
'Chrom_Chi0n'
'Chrom_Chi0v'
'Chrom_Chi1'
'Chrom_Chi1n'
'Chrom_Chi1v'
'Chrom_Chi2n'
'Chrom_Chi2v'
'Chrom_Chi3n'
'Chrom_Chi3v'
'Chrom_Chi4n'
'Chrom_Chi4v'
'Chrom_HallKierAlpha'
'Chrom_Ipc'
'Chrom_Kappa1'
'Chrom_Kappa2'
'Chrom_Kappa3'
'Chrom_LabuteASA'
'Chrom_PEOE_VSA1'
'Chrom_PEOE_VSA10'
'Chrom_PEOE_VSA11'
'Chrom_PEOE_VSA12'
'Chrom_PEOE_VSA13'
'Chrom_PE

In [157]:
# ensuring all entries within the Deep4Chem descriptor dataset are both finite and valid - following few cells added due to errors obtained when 
# attempting to train/test the tree-based models on the dataset
print(np.isinf(X).sum().sort_values(ascending=False).head(20))
print(X.isna().sum().sort_values(ascending=False).head(20))

Chrom_MaxEStateIndex    0
Chrom_MinEStateIndex    0
Solv_SlogP_VSA5         0
Solv_SlogP_VSA4         0
Solv_SlogP_VSA3         0
Solv_SlogP_VSA2         0
Solv_SlogP_VSA12        0
Solv_SlogP_VSA11        0
Solv_SlogP_VSA10        0
Solv_SlogP_VSA1         0
Solv_SMR_VSA9           0
Solv_SMR_VSA8           0
Solv_SMR_VSA7           0
Solv_SMR_VSA6           0
Solv_SMR_VSA5           0
Solv_SMR_VSA4           0
Solv_SMR_VSA3           0
Solv_SMR_VSA2           0
Solv_SMR_VSA10          0
Solv_SMR_VSA1           0
dtype: int64
Chrom_MaxEStateIndex    0
Chrom_MinEStateIndex    0
Solv_SlogP_VSA5         0
Solv_SlogP_VSA4         0
Solv_SlogP_VSA3         0
Solv_SlogP_VSA2         0
Solv_SlogP_VSA12        0
Solv_SlogP_VSA11        0
Solv_SlogP_VSA10        0
Solv_SlogP_VSA1         0
Solv_SMR_VSA9           0
Solv_SMR_VSA8           0
Solv_SMR_VSA7           0
Solv_SMR_VSA6           0
Solv_SMR_VSA5           0
Solv_SMR_VSA4           0
Solv_SMR_VSA3           0
Solv_SMR_VSA2           0

In [158]:
# as all values are technically valid and finite, investigating the if any values are of a particularly large magnitude => responsible for the errors
print('Largest value:', np.nanmax(X.values))
print('Smallest value:', np.nanmin(X.values))

Largest value: 8.335549151513545e+48
Smallest value: -104.0400000000001


In [159]:
# investigating largest values within the dataset, and corresponding descriptor names - 'Chrom_Ipc' feature likely causing the obtained errors due to 
# having a value of extremely large magnitude
largest = X.max().sort_values(ascending=False)
print(largest.head(20))

Chrom_Ipc                    8.335549e+48
Solv_Kappa3                  9.507960e+03
Chrom_BertzCT                7.628557e+03
Chrom_MolWt                  2.052522e+03
Chrom_ExactMolWt             2.050871e+03
Chrom_HeavyAtomMolWt         1.881865e+03
Solv_Ipc                     1.347912e+03
Chrom_LabuteASA              9.295924e+02
Chrom_NumValenceElectrons    8.400000e+02
Chrom_MolMR                  6.682800e+02
Chrom_EState_VSA5            6.457508e+02
Chrom_SlogP_VSA5             6.309625e+02
Chrom_PEOE_VSA6              6.152208e+02
Chrom_SMR_VSA5               6.005257e+02
Chrom_EState_VSA8            5.997538e+02
Chrom_SMR_VSA7               5.642035e+02
Chrom_SlogP_VSA6             5.095748e+02
Chrom_PEOE_VSA7              3.789556e+02
Chrom_VSA_EState1            3.775607e+02
Chrom_SMR_VSA10              3.770357e+02
dtype: float64


### Comments:

416 total descriptors created by the input SMILES strings (208 corresponding to both chromophores and solvents)

Following cleaning of the full Deep4Chem dataset, including removing invalid SMILES (solvent) input strings and rows which contained missing descriptor values, the number of entries was reduced from 8032 to 7996 (-36 rows)

The 'Chrom_Ipc' descriptor column values are particularly large - added additional limits to my tree-based model Python scripts to ensure descriptor columns containing values above a specified threshold were filtered prior to training

## Descriptor Generation for Deep4Chem Training Subset

In [160]:
# initialising a list to store descriptor vectors for all chromophores within the training subset
train_chromophore_desc = []

# iterating through every chromophore SMILES in the subset, and calculating and storing all corresponding descriptors for each molecule
for smiles in d4c_train['Chromophore']:
    train_chromophore_desc.append(calculate_descriptors(smiles))

# converting the chromophore descriptor lists into a DataFrame - prefixing each descriptor name with 'Chrom_' to distinguish chromophore descriptors 
# from solvent descriptors (calculated in the following cell)
train_chromophore_df = pd.DataFrame(train_chromophore_desc, columns=[f'Chrom_{d}' for d in descriptor_names])

In [161]:
# initialising a list to store descriptor vectors for all solvents within the training subset
train_solvent_desc = []

# iterating through every solvent SMILES in the subset, and calculating and storing all corresponding descriptors for each molecule
for smiles in d4c_train['Solvent']:
    train_solvent_desc.append(calculate_descriptors(smiles))

# converting the solvent descriptor lists into a DataFrame - prefixing each descriptor name with 'Solv_' to distinguish solvent descriptors from 
# chromophore descriptors (calculated in the previous cell)
train_solvent_df = pd.DataFrame(train_solvent_desc, columns=[f'Solv_{d}' for d in descriptor_names])

In [162]:
# concatenating both chromophore and solvent training descriptor DataFrames as a matrix - each column are the descriptor names for both chromophore and
# solvent inputs, and each row is the corresponding values calculated for each and every (chromophore and solvent) molecular descriptor
X_train = pd.concat([train_chromophore_df, train_solvent_df], axis=1)

In [163]:
# viewing the shape and first few values of the descriptor matrix to ensure all valid entries in the Deep4Chem training subset have been retained, and
# that the descriptors concatenated correctly - these values will be used as inputs in tree-based, non-Chemprop models (Random Forest, XGBoost, 
# LightGBM)
print(X_train.shape)
print(X_train.head())

(5959, 416)
   Chrom_MaxEStateIndex  Chrom_MinEStateIndex  Chrom_MaxAbsEStateIndex  \
0              2.120370              1.310185                 2.120370   
1              2.504398             -1.264830                 2.504398   
2              2.433496             -1.264353                 2.433496   
3              2.410962             -0.715627                 2.410962   
4              2.528548             -1.557930                 2.528548   

   Chrom_MinAbsEStateIndex  Chrom_qed  Chrom_MolWt  Chrom_HeavyAtomMolWt  \
0                 1.310185   0.511431      128.174               120.110   
1                 1.264830   0.689738      242.438               220.262   
2                 1.264353   0.726764      272.540               248.348   
3                 0.715627   0.710854      244.486               224.326   
4                 0.355418   0.556144      356.702               320.414   

   Chrom_ExactMolWt  Chrom_NumValenceElectrons  Chrom_NumRadicalElectrons  \
0        

In [164]:
# creating a list containing only the target property columns from the Deep4Chem training subset
y_train = d4c_train[['Absorption max (nm)', 'log(e/mol-1 dm3 cm-1)']]

In [165]:
# viewing the shape and first few values of this list to ensure all valid entries in the Deep4Chem test subset have been retained, and that the dataset
# was reduced correctly - these values will be used as targets in tree-based, non-Chemprop models (Random Forest, XGBoost, LightGBM)
print(y_train.shape)
print(y_train.head())

(5959, 2)
   Absorption max (nm)  log(e/mol-1 dm3 cm-1)
0                286.0                   3.58
1                294.0                   3.74
2                300.0                   3.87
3                300.0                   3.86
4                302.0                   3.91


In [166]:
# creating a Boolean mask that is 'False' for rows containing any missing values, and 'True' for rows with no missing values within the training subset
# descriptor matrix
mask = ~X_train.isna().any(axis=1)

# keeping only the rows of the training subset descriptor matrix and target properties with no missing descriptor values
X_train = X_train.loc[mask]
y_train = y_train.loc[mask]

# identically filtering the Deep4Chem training subset, ensuring it contains only molecules with valid descriptors and corresponding target values
d4c_train = d4c_train.loc[mask]

In [167]:
# viewing the lengths of the training subset descriptor matrix and target properties list to ensure they are the same length following data cleaning
print(len(X_train))
print(len(y_train))

5940
5940


In [168]:
# constructing the fully-featurised Deep4Chem training subset by concatenating the descriptor matrix and the measured target property values list - the
# first 416 columns correspond to the former's features, and the final 2 columns correspond to the latter's
descriptor_train_subset = pd.concat([X_train, y_train], axis=1)

# saving the featurised Deep4Chem training subset as a CSV file
descriptor_train_subset.to_csv('d4c_train_rdkit_descriptors.csv', index=False)

In [169]:
# viewing shape of the fully-featurised Deep4Chem training subset
print('Shape:', descriptor_train_subset.shape)

Shape: (5940, 418)


## Descriptor Generation for Deep4Chem Test Subset

In [170]:
# initialising a list to store descriptor vectors for all chromophores within the test subset
test_chromophore_desc = []

# iterating through every chromophore SMILES in the subset, and calculating and storing all corresponding descriptors for each molecule
for smiles in d4c_test['Chromophore']:
    test_chromophore_desc.append(calculate_descriptors(smiles))

# converting the chromophore descriptor lists into a DataFrame - prefixing each descriptor name with 'Chrom_' to distinguish chromophore descriptors 
# from solvent descriptors (calculated in the following cell)
test_chromophore_df = pd.DataFrame(test_chromophore_desc, columns=[f'Chrom_{d}' for d in descriptor_names])

In [171]:
# initialising a list to store descriptor vectors for all solvents within the test subset
test_solvent_desc = []

# iterating through every solvent SMILES in the subset, and calculating and storing all corresponding descriptors for each molecule
for smiles in d4c_test['Solvent']:
    test_solvent_desc.append(calculate_descriptors(smiles))

# converting the solvent descriptor lists into a DataFrame - prefixing each descriptor name with 'Solv_' to distinguish solvent descriptors from 
# chromophore descriptors (calculated in the previous cell)
test_solvent_df = pd.DataFrame(test_solvent_desc, columns=[f'Solv_{d}' for d in descriptor_names])

In [172]:
# concatenating both chromophore and solvent test descriptor DataFrames as a matrix - each column are the descriptor names for both chromophore and 
# solvent inputs, and each row is the corresponding values calculated for each and every (chromophore and solvent) molecular descriptor
X_test = pd.concat([test_chromophore_df, test_solvent_df], axis=1)

In [173]:
# viewing the shape and first few values of the descriptor matrix to ensure all valid entries in the Deep4Chem test subset have been retained, and that
# the descriptors concatenated correctly - these values will be used as inputs in tree-based, non-Chemprop models (Random Forest, XGBoost, LightGBM)
print(X_test.shape)
print(X_test.head())

(2059, 416)
   Chrom_MaxEStateIndex  Chrom_MinEStateIndex  Chrom_MaxAbsEStateIndex  \
0              2.397593             -1.197755                 2.397593   
1              2.376759             -0.679074                 2.376759   
2              2.480926             -1.445440                 2.480926   
3              2.542476             -1.312351                 2.542476   
4              9.090650             -1.342852                 9.090650   

   Chrom_MinAbsEStateIndex  Chrom_qed  Chrom_MolWt  Chrom_HeavyAtomMolWt  \
0                 1.197755   0.619051      200.357               184.229   
1                 0.679074   0.600400      186.330               172.218   
2                 0.386036   0.643330      242.438               220.262   
3                 1.312351   0.406870      298.546               268.306   
4                 0.775556   0.683403      225.367               210.247   

   Chrom_ExactMolWt  Chrom_NumValenceElectrons  Chrom_NumRadicalElectrons  \
0        

In [174]:
# creating a list containing only the target property columns from the Deep4Chem test subset
y_test = d4c_test[['Absorption max (nm)', 'log(e/mol-1 dm3 cm-1)']]

In [175]:
# viewing the shape and first few values of this list to ensure all valid entries in the Deep4Chem test subset have been retained, and that the dataset
# was reduced correctly - these values will be used as targets in tree-based, non-Chemprop models (Random Forest, XGBoost, LightGBM)
print(y_test.shape)
print(y_test.head())

(2059, 2)
   Absorption max (nm)  log(e/mol-1 dm3 cm-1)
0                294.0                   3.73
1                294.0                   3.75
2                295.0                   3.78
3                294.0                   3.76
4                315.0                   3.88


In [176]:
# creating a Boolean mask that is 'False' for rows containing any missing values, and 'True' for rows with no missing values within the test subset
# descriptor matrix
mask = ~X_test.isna().any(axis=1)

# keeping only the rows of the test subset descriptor matrix and target properties with no missing descriptor values
X_test = X_test.loc[mask]
y_test = y_test.loc[mask]

# identically filtering the Deep4Chem test subset, ensuring it contains only molecules with valid descriptors and corresponding target values
d4c_test = d4c_test.loc[mask]

In [177]:
# viewing the lengths of the test subset descriptor matrix and target properties list to ensure they are the same length following data cleaning
print(len(X_test))
print(len(y_test))

2056
2056


In [178]:
# constructing the fully-featurised Deep4Chem test subset by concatenating the descriptor matrix and the measured target property values list - the 
# first 416 columns correspond to the former's features, and the final 2 columns correspond to the latter's
descriptor_test_subset = pd.concat([X_test, y_test], axis=1)

# saving the featurised Deep4Chem test subset as a CSV file
descriptor_test_subset.to_csv('d4c_test_rdkit_descriptors.csv', index=False)

In [179]:
# viewing shape of the fully-featurised Deep4Chem test subset
print('Shape:', descriptor_test_subset.shape)

Shape: (2056, 418)


## Removing Redundant/Correlated Features

In [180]:
# loading descriptor Deep4Chem subsets
train_desc = pd.read_csv('d4c_train_rdkit_descriptors.csv')
test_desc = pd.read_csv('d4c_test_rdkit_descriptors.csv')

# separating descriptor and target columns in the training descriptor subset
desc_cols = train_desc.columns[:-2]
target_cols = train_desc.columns[-2:]

# retaining descriptor columns for training and test subset DataFrames for correlation analysis
X_train_desc = train_desc[desc_cols]
X_test_desc = test_desc[desc_cols]

In [181]:
# calculating correlation matrix using the training descriptors (same as in test subset)
corr_matrix = X_train_desc.corr().abs()

# extracting upper triangle of correlation matrix - keeping only one half of the correlation matrix as it's symmetric => avoids checking each 
# descriptor pair twice or comparing descriptors with themselves
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# specifying feature correlation threshold value
corr_threshold = 0.8

# specifying columns to drop from datasets based on the given correlation threshold - retains one representative descriptor from each correlated group
drop_cols = [column for column in upper.columns if any(upper[column] >= corr_threshold)]

# printing number of and names of columns to be dropped
print(f'{len(drop_cols)} descriptors will be removed.')
print(drop_cols)

# dropping highly-correlated descriptors from both training and test subsets
X_train_desc = X_train_desc.drop(columns=drop_cols)
X_test_desc = X_test_desc.drop(columns=drop_cols)

150 descriptors will be removed.
['Chrom_MaxAbsEStateIndex', 'Chrom_HeavyAtomMolWt', 'Chrom_ExactMolWt', 'Chrom_NumValenceElectrons', 'Chrom_MaxAbsPartialCharge', 'Chrom_MinAbsPartialCharge', 'Chrom_FpDensityMorgan2', 'Chrom_FpDensityMorgan3', 'Chrom_BCUT2D_LOGPLOW', 'Chrom_BCUT2D_MRHI', 'Chrom_BertzCT', 'Chrom_Chi0', 'Chrom_Chi0n', 'Chrom_Chi0v', 'Chrom_Chi1', 'Chrom_Chi1n', 'Chrom_Chi1v', 'Chrom_Chi2n', 'Chrom_Chi2v', 'Chrom_Chi3n', 'Chrom_Chi3v', 'Chrom_Chi4n', 'Chrom_Chi4v', 'Chrom_HallKierAlpha', 'Chrom_Kappa1', 'Chrom_Kappa2', 'Chrom_Kappa3', 'Chrom_LabuteASA', 'Chrom_PEOE_VSA6', 'Chrom_PEOE_VSA7', 'Chrom_SMR_VSA7', 'Chrom_SlogP_VSA5', 'Chrom_SlogP_VSA6', 'Chrom_VSA_EState1', 'Chrom_VSA_EState6', 'Chrom_HeavyAtomCount', 'Chrom_NOCount', 'Chrom_NumAliphaticRings', 'Chrom_NumAromaticCarbocycles', 'Chrom_NumAromaticRings', 'Chrom_NumHAcceptors', 'Chrom_NumHDonors', 'Chrom_NumHeteroatoms', 'Chrom_NumRotatableBonds', 'Chrom_RingCount', 'Chrom_MolLogP', 'Chrom_MolMR', 'Chrom_fr_Ar_N', 

In [182]:
# recombining filtered descriptor columns with target property columns to create full, filtered descriptor Deep4Chem descriptor subsets
train_desc_filtered = pd.concat([X_train_desc, train_desc[target_cols]], axis=1)
test_desc_filtered = pd.concat([X_test_desc, test_desc[target_cols]], axis=1)

# saving filtered descriptor datasets as CSV files
train_desc_filtered.to_csv('d4c_train_rdkit_descriptors_filtered.csv', index=False)
test_desc_filtered.to_csv('d4c_test_rdkit_descriptors_filtered.csv', index=False)

# printing number of remaining descriptors after filtering
print(f'Remaining descriptors: {X_train_desc.shape[1]}')

Remaining descriptors: 266


### Comments:

150 descriptors were removed from the featurised Deep4Chem subsets - most were solvent descriptors but relatively equal => good balance of chromophore and solvent input descriptors retained following filtration for hyperparameter optimisation and training/prediction 